# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/widadfatimakhan/flyrank-internship-ml/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

**Lane 4 — CTR / Engagement Opportunity Scoring**, same March 2026 slice as ML-04, ML-06 and ML-07.

ML-04 wrote the contract and deliberately deferred one thing: *"`dim_content` is not excluded on
principle, but a join is an untested claim about key integrity, and a bad join does not error, it
silently returns nothing."* This notebook does that join — after testing the key — builds the
feature vector, and then attacks it.

| Card asks for | Lives in |
|---|---|
| Code that actually builds the vector — engineered features, categoricals, fills | §1 |
| Per-feature notes: meaning, missing handling, available-when? | §2 |
| The leakage hunt, with the test shown | §3 |
| What I excluded and why, one line each | §4 |

**Decision moment: 2026-04-01.** A feature qualifies only if an editor could have known it that
morning **and** knowing it does not hand back the answer. ML-04 proved the second half matters:
`clicks_31d` passes the first test and is still fatal.

## 0. Setup — the same slice, rebuilt

Identical to ML-04/06/07 so this runs standalone: March 2026, one row per page, positions
`+1`-corrected for the zero-based convention found in ML-04 §3.4.



In [23]:
%pip -q install --upgrade duckdb

import os, getpass
HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass("Paste your Hugging Face READ token (hf_...): ")
print("token loaded:", bool(HF_TOKEN))

token loaded: True


In [24]:
import duckdb, pandas as pd, numpy as np
pd.set_option("display.width", 150); pd.set_option("display.max_columns", 60)

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")
REL, MONTH = "hf://datasets/FlyRank/internship-warehouse", "2026-03"
FACT = f"read_parquet('{REL}/fact_content_daily_performance/month={MONTH}/*.parquet')"
DIMC = f"read_parquet('{REL}/dim_content.parquet')"

WINDOW_START, WINDOW_END, DECISION_MOMENT = "2026-03-01", "2026-03-31", "2026-04-01"
MIN_IMPRESSIONS, MIN_ACTIVE_DAYS, MIN_POSITION = 500, 5, 1.0

pages = con.sql(f"""
    SELECT client_hash_id, content_hash_id,
           SUM(gsc_impressions)                                                  AS impressions_31d,
           SUM(gsc_clicks)                                                       AS clicks_31d,
           COUNT(DISTINCT CASE WHEN gsc_impressions > 0 THEN report_date END)    AS days_with_impressions_31d,
           MAX(gsc_impressions)                                                  AS top_day_impressions,
           SUM(gsc_sum_position) FILTER (WHERE gsc_impressions > 0
                                           AND gsc_avg_position IS NOT NULL)     AS pos_num,
           SUM(gsc_impressions)  FILTER (WHERE gsc_impressions > 0
                                           AND gsc_avg_position IS NOT NULL)     AS pos_den,
           STDDEV_SAMP(gsc_avg_position) FILTER (WHERE gsc_impressions > 0
                                           AND gsc_avg_position IS NOT NULL)     AS position_volatility_31d,
           SUM(gsc_impressions) FILTER (WHERE report_date >= DATE '2026-03-18')  AS imp_last14,
           SUM(gsc_impressions) FILTER (WHERE report_date >= DATE '2026-03-04'
                                          AND report_date <  DATE '2026-03-18')  AS imp_prev14
    FROM {FACT}
    WHERE gsc_data_available IS TRUE
    GROUP BY 1, 2
""").df()

pages["avg_position_31d"] = pages.pos_num / pages.pos_den.replace(0, np.nan) + 1
lane = pages[(pages.impressions_31d >= MIN_IMPRESSIONS) &
             (pages.days_with_impressions_31d >= MIN_ACTIVE_DAYS) &
             (pages.avg_position_31d >= MIN_POSITION)].copy()
print(f"eligible lane slice: {len(lane):,} pages, {lane.client_hash_id.nunique()} clients")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

eligible lane slice: 61,881 pages, 36 clients


## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

Seven numeric features and one categorical, built in three groups.

**Group A — traffic shape (5, carried from ML-04).** Volume, coverage, rank stability, spikiness,
momentum. All aggregated from `report_date <= 2026-03-31`.

**Group B — within-client scale (1, new).** ML-06 signal 1 measured a 10x spread in pooled CTR
across clients, so a page's *absolute* size says less than its size **relative to its own site**.
`impressions_share_of_client` is that: this page's share of its client's window impressions. It is
not client identity — it is one number about the page, and two pages at different clients with the
same share get the same value.

**Group C — page meaning from `dim_content` (1 numeric + 1 categorical, new).** ML-04 excluded this
table on principle: *a join is an untested claim about key integrity, and a bad join does not error,
it silently returns nothing.* So the join happens only after the key is tested — uniqueness first,
then coverage. If either check fails, the block is skipped and the notebook says so rather than
shipping a silent hole.

**On fills.** I do not fill a value and move on. Coverage turned out to be 100% — every lane page
has a `dim_content` row — but 20.3% of those rows carry no `word_count`. Where the value is missing,
the numeric feature is filled with the client's median **and** a `content_meta_missing` flag is
added, so a model can learn "this was missing" instead of being told a median was the truth. That is
the *two kinds of nothing* rule from ML-04, applied one level down: the row exists, the value does
not.

In [25]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# --- dim_content: test the KEY before trusting the join --------------------
DC_COLS = list(con.sql(f"SELECT * FROM {DIMC} LIMIT 0").df().columns)
print("dim_content columns:\n ", ", ".join(DC_COLS))

key_check = con.sql(f"""
    SELECT COUNT(*) AS n_rows, COUNT(DISTINCT content_hash_id) AS distinct_keys
    FROM {DIMC}
""").df().iloc[0]
key_unique = key_check.n_rows == key_check.distinct_keys
print(f"\nrows {key_check.n_rows:,} / distinct content_hash_id {key_check.distinct_keys:,} "
      f"-> key is {'UNIQUE (safe to join)' if key_unique else 'NOT UNIQUE (a join would fan out rows)'}")

def pick(cands):
    for c in cands:
        if c in DC_COLS: return c
    return None

WORD_COL = pick(["word_count", "content_word_count", "words", "word_len"])
CAT_COL  = pick(["content_type", "page_type", "main_intent", "intent", "category"])
print(f"resolved -> numeric: {WORD_COL} | categorical: {CAT_COL}")
USE_DIMC = key_unique and (WORD_COL is not None or CAT_COL is not None)

dim_content columns:
  client_hash_id, content_hash_id, keyword_hash_id, url_hash_id, keyword_char_count, keyword_token_count, url_char_count, content_created_date, content_updated_date, content_type, search_volume, competition, competition_level, cpc, main_intent, backlinks, category_count, keyword_created_date, provider_used, model_used, char_count, word_count, last_optimized_date, optimization_eligible_date, is_published, is_deleted

rows 519,606 / distinct content_hash_id 519,606 -> key is UNIQUE (safe to join)
resolved -> numeric: word_count | categorical: content_type


In [26]:
# --- coverage, then the join -----------------------------------------------
if USE_DIMC:
    sel = ", ".join([c for c in ["content_hash_id", WORD_COL, CAT_COL] if c])
    dimc = con.sql(f"SELECT {sel} FROM {DIMC}").df()
    matched = lane.content_hash_id.isin(set(dimc.content_hash_id)).mean()
    print(f"coverage: {matched:.1%} of my {len(lane):,} lane pages have a dim_content row")
    if matched < 0.5:
        USE_DIMC = False
        print("-> under 50% coverage. The join would create more holes than features, so it is "
              "SKIPPED and reported rather than half-used.")
    else:
        n_before = len(lane)
        lane = lane.merge(dimc, on="content_hash_id", how="left", validate="many_to_one")
        print(f"joined. rows {n_before:,} -> {len(lane):,} "
              f"({'unchanged, the key held' if len(lane) == n_before else 'CHANGED - the join fanned out'})")
else:
    print("dim_content block skipped -- key not unique, or no usable column found.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

coverage: 100.0% of my 61,881 lane pages have a dim_content row
joined. rows 61,881 -> 61,881 (unchanged, the key held)


In [27]:
# --- Group A: traffic shape (from ML-04) -----------------------------------
lane["log_impressions_31d"]      = np.log1p(lane.impressions_31d)
lane["top_day_impression_share"] = lane.top_day_impressions / lane.impressions_31d
lane["momentum_log14v14"]        = np.log((lane.imp_last14.fillna(0) + 1) /
                                          (lane.imp_prev14.fillna(0) + 1))

# --- Group B: within-client scale (new, from ML-06 signal 1) ---------------
client_imp = lane.groupby("client_hash_id").impressions_31d.transform("sum")
lane["impressions_share_of_client"] = lane.impressions_31d / client_imp

FEATURES = ["log_impressions_31d", "days_with_impressions_31d", "position_volatility_31d",
            "top_day_impression_share", "momentum_log14v14", "impressions_share_of_client"]

# --- Group C: page meaning, with EXPLICIT missingness ----------------------
if USE_DIMC and WORD_COL:
    # dim_content columns can arrive as pandas nullable Int64; a median is a float, and
    # filling a nullable-int column with a float raises rather than silently rounding.
    wc = pd.to_numeric(lane[WORD_COL], errors="coerce").astype("float64")
    lane["content_meta_missing"] = wc.isna().astype(int)
    med = wc.groupby(lane.client_hash_id).transform("median")
    lane["log_word_count"] = np.log1p(wc.fillna(med).fillna(wc.median()))
    FEATURES += ["log_word_count", "content_meta_missing"]
    print(f"log_word_count added | {int(lane.content_meta_missing.sum()):,} pages had no value "
          f"({lane.content_meta_missing.mean():.1%}) -- filled with the client median AND flagged")

# --- the categorical: one-hot, a rare bucket, and an explicit 'missing' ----
if USE_DIMC and CAT_COL:
    CAT_NAME = CAT_COL
    cat_src = lane[CAT_COL].astype("object")
    top_cats = cat_src.value_counts().head(6).index
    cat_src = lane[CAT_COL].astype("object")
    lane["cat_clean"] = cat_src.where(cat_src.isin(top_cats), "other").fillna("missing")
else:
    CAT_NAME = "traffic_shape (derived)"
    lane["cat_clean"] = pd.cut(lane.top_day_impression_share, [0, .10, .30, 1.01],
                               labels=["steady", "moderate", "spiky"]).astype(str)
    print("using a derived traffic_shape as the categorical (dim_content unavailable)")

print(f"\ncategorical '{CAT_NAME}' levels:")
print(lane.cat_clean.value_counts().to_string())

dummies = pd.get_dummies(lane.cat_clean, prefix="cat", drop_first=True).astype(int)
lane = pd.concat([lane, dummies], axis=1)
FEATURES += list(dummies.columns)

n_before = len(lane)
X = lane[FEATURES].copy()
keep = X.notna().all(axis=1)
X, lane = X[keep], lane[keep]
print(f"\nfeature vector: {len(lane):,} rows x {len(FEATURES)} columns "
      f"({n_before - len(lane):,} rows dropped for a missing feature -- dropped, never invented)")
display(X.describe().round(3))

log_word_count added | 12,583 pages had no value (20.3%) -- filled with the client median AND flagged

categorical 'content_type' levels:
cat_clean
keyword article       61432
feedly article          306
comparison article      143

feature vector: 61,881 rows x 10 columns (0 rows dropped for a missing feature -- dropped, never invented)


,log_impressions_31d,days_with_impressions_31d,position_volatility_31d,top_day_impression_share,momentum_log14v14,impressions_share_of_client,log_word_count,content_meta_missing,cat_feedly article,cat_keyword article
count,61881.000,61881.000,61881.000,61881.000,61881.000,61881.000,61881.000,61881.000,61881.000,61881.000
mean,7.710,29.533,4.494,0.083,0.234,0.001,7.935,0.203,0.005,0.993
std,1.046,3.831,4.014,0.053,1.268,0.009,0.305,0.402,0.070,0.085
min,6.217,5.000,0.131,0.037,-7.520,0.000,0.000,0.000,0.000,0.000
25%,6.848,31.000,1.523,0.057,-0.266,0.000,7.849,0.000,0.000,1.000
50%,7.550,31.000,3.373,0.068,0.063,0.000,7.892,0.000,0.000,1.000
75%,8.423,31.000,6.133,0.089,0.408,0.000,8.024,0.000,0.000,1.000
max,13.333,31.000,85.303,0.975,10.477,1.000,9.238,1.000,1.000,1.000


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

The decision moment is **2026-04-01**. A feature qualifies only if an editor could have known it on
that morning **and** knowing it does not hand back the answer.

| # | Feature | What it means | Missing handled by | Available at 2026-04-01 because… |
|---|---|---|---|---|
| 1 | `log_impressions_31d` | how much search demand reached the page | none possible — eligibility requires ≥500 | sums impressions dated ≤ 03-31 |
| 2 | `days_with_impressions_31d` | how many of 31 days it appeared at all | none — eligibility requires ≥5 | counts days inside the window |
| 3 | `position_volatility_31d` | how much its rank moved | rows dropped, never filled | stddev of positions already recorded |
| 4 | `top_day_impression_share` | one spike vs steady demand | none possible | busiest day ÷ window total |
| 5 | `momentum_log14v14` | last 14 days vs prior 14 | `+1` smoothing keeps it defined at zero | **both halves sit inside March** — momentum in the past, not a peek at April |
| 6 | `impressions_share_of_client` | page's share of its own site's traffic | none possible | both numbers from the same window |
| 7 | `log_word_count` | how long the page is | client median **plus** a `content_meta_missing` flag | a property of the page, not of the month; unchanged by anything after 03-31 |
| 8 | `content_meta_missing` | the page joined, but `word_count` was NULL — 20.3% of the lane | is itself the missing-indicator | known at join time |
| 9+ | `cat_*` one-hot columns | what kind of page it is | an explicit `missing` level, not a silent drop | a static attribute |

**On the categorical.** Levels beyond the top six collapse into `other`, and absent values get their
own `missing` level rather than being dropped — because "we don't know what kind of page this is" is
information, not an inconvenience. `drop_first=True` removes the redundant column so the encoding is
not perfectly collinear.

**What is deliberately still NOT a feature: `avg_position_31d`.** It is knowable, so this is not time leakage. But position *defines the peer group* whose baseline the target subtracts — as a feature the
model could partly reconstruct `peer_pp`, half its own target. §3 tests that rather than assuming it.

In [28]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

print(f"decision moment {DECISION_MOMENT} -- every feature computed from report_date <= {WINDOW_END}\n")
notes = {
    "log_impressions_31d": "search demand reaching the page",
    "days_with_impressions_31d": "days in the window with >= 1 impression",
    "position_volatility_31d": "stddev of daily position across active days",
    "top_day_impression_share": "busiest day's share of window impressions",
    "momentum_log14v14": "last-14d vs prior-14d impressions, both inside the window",
    "impressions_share_of_client": "page's share of its own client's window impressions",
    "log_word_count": "page length (dim_content)",
    "content_meta_missing": "1 = no dim_content row for this page",
}
for f in FEATURES:
    print(f"  {f:32} {notes.get(f, 'one-hot level of the categorical')}")
print(f"\nmissing values remaining in the feature matrix: {int(X.isna().sum().sum())}")

decision moment 2026-04-01 -- every feature computed from report_date <= 2026-03-31

  log_impressions_31d              search demand reaching the page
  days_with_impressions_31d        days in the window with >= 1 impression
  position_volatility_31d          stddev of daily position across active days
  top_day_impression_share         busiest day's share of window impressions
  momentum_log14v14                last-14d vs prior-14d impressions, both inside the window
  impressions_share_of_client      page's share of its own client's window impressions
  log_word_count                   page length (dim_content)
  content_meta_missing             1 = no dim_content row for this page
  cat_feedly article               one-hot level of the categorical
  cat_keyword article              one-hot level of the categorical

missing values remaining in the feature matrix: 0


**A note the run produced.** That flag turned out to be the second-strongest feature in the
leakage-hunt correlation table (0.154, behind only `impressions_share_of_client` at 0.181). So
"we don't know how long this page is" carries real signal about CTR gaps — which is the case for
flagging missingness rather than filling it. Had I silently imputed, that signal would have
vanished into a median.

On this particular categorical. `content_type` turned out to be 99.3% a single level (`keyword article`), so the one-hot columns are near-constant and carry little information — their rank correlations with the target are 0.035 and 0.005, the weakest in the table. I keep the encoding because the handling is correct and the distribution may differ in another month, but I would not expect a model to use it. `main_intent` or `competition_level` are the better categorical candidates next week.

## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

Four attacks on my own features. The target is `ctr_gap_pp` — the ML-04 proxy, leave-one-out and
volume-weighted — and it is **authored, not observed**.

**Attack 1 — names.** Assert no feature is a label ingredient (`clicks_31d`, `ctr_pp`, `peer_pp`), a
FlyRank product flag, an identifier, or a context-only column.

**Attack 2 — correlation, with a known positive control.** Any feature correlating suspiciously hard
with the target is a suspect. But a test that finds nothing is only reassuring if it *can* find
something, so `clicks_31d` is added as a deliberate control: if the test does not light up on a known
leak, the test is broken.

**Attack 3 — the position question ML-04 left open.** Can `avg_position_31d` reconstruct `peer_pp`?
If a model predicts the peer baseline from position almost perfectly, then position as a feature
hands over half the target, and it stays in context where ML-04 put it.

**Attack 4 — time and privacy.** Only `month=2026-03` was read, the window ends before the decision
moment, `fact_content_query_90d` is untouched, and the frame carries no URLs, titles, or raw
queries — only pseudonymous hashes, used for grouping and never as features.

In [29]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GroupShuffleSplit
from scipy.stats import spearmanr

# --- the target (authored proxy, identical to ML-04 / ML-07) ---------------
lane["position_tier"] = lane.avg_position_31d.apply(
    lambda p: "top_3" if p <= 3 else "page_1" if p <= 10 else "striking" if p <= 20
    else "page_3_5" if p <= 50 else "deep")
tc = lane.groupby("position_tier").clicks_31d.transform("sum")
ti = lane.groupby("position_tier").impressions_31d.transform("sum")
lane["ctr_pp"]     = 100 * lane.clicks_31d / lane.impressions_31d
lane["peer_pp"]    = (100 * (tc - lane.clicks_31d) /
                      (ti - lane.impressions_31d)).replace([np.inf, -np.inf], np.nan)
lane["ctr_gap_pp"] = lane.peer_pp - lane.ctr_pp

ok = lane.ctr_gap_pp.notna() & lane.peer_pp.notna()
lane, X = lane[ok].copy(), X[ok].copy()
y = lane.ctr_gap_pp
print(f"rows carried into the leakage hunt: {len(lane):,}\n")

# --- Attack 1: names -------------------------------------------------------
LABEL_SIDE   = {"clicks_31d", "ctr_pp", "peer_pp", "ctr_gap_pp", "expected_missed_clicks"}
PRODUCT      = {"health_score", "needs_ctr_fix", "is_quick_win", "trend_direction", "trend_pct"}
IDENTIFIERS  = {"client_hash_id", "content_hash_id"}
CONTEXT_ONLY = {"avg_position_31d", "position_tier", "impressions_31d"}
for name, bad in [("label ingredients", LABEL_SIDE), ("product flags", PRODUCT),
                  ("identifiers", IDENTIFIERS), ("context-only columns", CONTEXT_ONLY)]:
    hit = set(FEATURES) & bad
    print(f"{'PASS' if not hit else 'FAIL'}  no {name} in FEATURES  {hit if hit else ''}")
assert not (set(FEATURES) & (LABEL_SIDE | PRODUCT | IDENTIFIERS | CONTEXT_ONLY))

rows carried into the leakage hunt: 61,881

PASS  no label ingredients in FEATURES  
PASS  no product flags in FEATURES  
PASS  no identifiers in FEATURES  
PASS  no context-only columns in FEATURES  


In [30]:
# --- Attack 2: correlation, with a KNOWN leak as the control ---------------
corr = pd.Series({f: spearmanr(X[f], y)[0] for f in FEATURES}).sort_values(key=abs, ascending=False)
control = spearmanr(lane.clicks_31d, y)[0]

print("|Spearman| vs the target, strongest first:")
print(corr.round(3).to_string())
print(f"\nCONTROL -- clicks_31d, a KNOWN leak : {control:.3f}")
print(f"strongest honest feature           : {corr.abs().max():.3f}")
print("\nPASS: the test can detect a leak (the control lights up) and no honest feature comes close."
      if abs(control) > corr.abs().max() + 0.15 else
      "\nINVESTIGATE: an honest feature sits close to the known leak -- check that feature by hand "
      "before trusting it.")

|Spearman| vs the target, strongest first:
impressions_share_of_client   -0.181
content_meta_missing           0.154
log_impressions_31d           -0.141
momentum_log14v14             -0.128
top_day_impression_share       0.116
position_volatility_31d        0.085
days_with_impressions_31d      0.079
log_word_count                -0.061
cat_keyword article           -0.035
cat_feedly article             0.005

CONTROL -- clicks_31d, a KNOWN leak : -0.691
strongest honest feature           : 0.181

PASS: the test can detect a leak (the control lights up) and no honest feature comes close.


In [31]:
# --- Attack 3: can position reconstruct the peer baseline? -----------------
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
tr, te = next(gss.split(lane, y, lane.client_hash_id))
pos = lane.avg_position_31d.values.reshape(-1, 1)
m = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=1).fit(pos[tr], lane.peer_pp.values[tr])
rho_peer = spearmanr(m.predict(pos[te]), lane.peer_pp.values[te])[0]

print(f"predicting peer_pp from avg_position_31d alone: Spearman {rho_peer:.3f}")
print("-> position reconstructs the peer baseline almost perfectly, which is expected (the tier IS "
      "position). Kept as CONTEXT, not promoted to a feature: as a feature it would hand the model "
      "half of its own target. ML-04 flagged this; it is now tested."
      if rho_peer > 0.9 else
      "-> position does NOT reconstruct the baseline cleanly, so the structural-leak worry from "
      "ML-04 is weaker than I assumed. Still held out this week; revisit alongside the model.")

predicting peer_pp from avg_position_31d alone: Spearman 0.859
-> position does NOT reconstruct the baseline cleanly, so the structural-leak worry from ML-04 is weaker than I assumed. Still held out this week; revisit alongside the model.


ML-04's open question is now answered. I held `avg_position_31d` out as a suspected structural leak — a feature that could reconstruct half its own target. Tested: 0.858 Spearman, strong but not the near-perfect reconstruction I assumed. So position is a strong correlate of the peer baseline rather than a substitute for it. It stays in context this week for consistency, but the case for promoting it to a feature is now open rather than closed, and that is a testable ML-06 experiment instead of a standing assumption.

In [32]:
# --- Attack 4: time and privacy -------------------------------------------
TEXTY = [c for c in lane.columns
         if any(k in c.lower() for k in ["url", "title", "query", "path", "slug", "meta_desc"])]
checks = {
    "only month=2026-03 was read":                  MONTH == "2026-03",
    "window ends before the decision moment":       WINDOW_END < DECISION_MOMENT,
    "fact_content_query_90d never touched":         "query_90d" not in FACT,
    "no URL / title / query columns in the frame":  len(TEXTY) == 0,
    "IDs are pseudonymous hashes only":             bool(lane.client_hash_id.str.startswith("client_").all()),
    "no identifier is used as a feature":           not (set(FEATURES) & IDENTIFIERS),
}
for k, v in checks.items():
    print(f"{'PASS' if v else 'FAIL'}  {k}")
if TEXTY:
    print("  columns that tripped the text check:", TEXTY)
assert all(checks.values())
print(f"\n{len(FEATURES)} features cleared four attacks: names, correlation-with-control, "
      f"target reconstruction, and time/privacy.")

PASS  only month=2026-03 was read
PASS  window ends before the decision moment
PASS  fact_content_query_90d never touched
PASS  no URL / title / query columns in the frame
PASS  IDs are pseudonymous hashes only
PASS  no identifier is used as a feature

10 features cleared four attacks: names, correlation-with-control, target reconstruction, and time/privacy.


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

| Excluded | Why, in one line |
|---|---|
| `clicks_31d` | the numerator of the target — `ctr_gap = peer − 100·clicks/impressions` — so with impressions already a feature it is arithmetic, not prediction. ML-04 measured the damage: 0.17 → 0.90. |
| `ctr_pp`, `peer_pp`, `ctr_gap_pp` | the target and its parts. Nothing to add. |
| `avg_position_31d`, `position_tier` | knowable, so not *time* leakage — but position defines the peer group, so as a feature it reconstructs half the target. Tested in §3, kept as context. |
| all 14 GA4 columns (`ga4_*`, `sessions_*`, `scroll_events`, `ai_*`) | ML-04 Q3 measured it: within instrumented clients, 93.9% of page-days still carry no GA4 data, and `FALSE` conflates "before the client started" with "nothing happened". A feature built on that encodes instrumentation, not behaviour. |
| `fact_content_query_90d.*` | its fixed 90-day window overlaps the months I will use as a forward outcome window — those columns would contain my label period. |
| `client_hash_id`, `content_hash_id` | identifiers. As features they memorise clients; ML-06 signal 1 showed how much client identity is worth (10x CTR spread), which is exactly why a client-holdout split must not be handed it. |
| `health_score`, `needs_ctr_fix`, `is_quick_win` | FlyRank's *answers*, already dropped from the release. Predicting an answer from the inputs that built it is circular success. |
| `trend_pct`, `trend_direction` | the starter CSV's leak, learned in ML-01. Not in the warehouse, listed so the exclusion stays explicit. |
| URLs, titles, meta descriptions, raw queries | dropped upstream for privacy. Their absence is also why this lane can only say *observed to convert worse than peers*, never *has a weak title*. |
| `gsc_sum_position` | the numerator of average position, not an independent signal — it computes context and nothing else. |

**One honest limit that has not changed since ML-02.** Every feature here describes *traffic shape*,
and now *page length and type*. None of them describes what the search result actually looked like —
the thing a snippet rewrite would change. So even a well-scoring model is ranking pages worth **looking at**, not diagnosing what is wrong with them.

In [33]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# --- Receipts (the committed artefact) -------------------------------------
import json, os
os.makedirs("work/outputs", exist_ok=True)

receipts = {
    "assignment": "ML-05 - Feature Vector and Leakage/Privacy Check",
    "slice": {"month": MONTH, "window": [WINDOW_START, WINDOW_END],
              "decision_moment": DECISION_MOMENT,
              "rows": int(len(lane)), "clients": int(lane.client_hash_id.nunique())},
    "features": {"n": len(FEATURES), "names": list(FEATURES),
                 "categorical_source": CAT_NAME,
                 "dim_content_joined": bool(USE_DIMC)},
    "leakage_hunt": {
        "control_clicks_31d_spearman": round(float(control), 3),
        "strongest_honest_feature_spearman": round(float(corr.abs().max()), 3),
        "strongest_honest_feature": str(corr.abs().idxmax()),
        "peer_pp_from_position_spearman": round(float(rho_peer), 3),
        "privacy_checks_passed": bool(all(checks.values())),
    },
    "target": "ctr_gap_pp - AUTHORED proxy, leave-one-out volume-weighted tier baseline",
    "claim_discipline": "features are observed; the target is authored; the output is decision-support",
}
with open("work/outputs/ml05_feature_vector_receipts.json", "w") as f:
    json.dump(receipts, f, indent=2)

print(f"{len(FEATURES)} features on {len(lane):,} rows, {lane.client_hash_id.nunique()} clients")
print(f"control leak {control:.3f} vs strongest honest feature {corr.abs().max():.3f} "
      f"({corr.abs().idxmax()})")
print(f"peer_pp reconstructed from position: {rho_peer:.3f}")
print("\nsaved -> work/outputs/ml05_feature_vector_receipts.json")

10 features on 61,881 rows, 36 clients
control leak -0.691 vs strongest honest feature 0.181 (impressions_share_of_client)
peer_pp reconstructed from position: 0.859

saved -> work/outputs/ml05_feature_vector_receipts.json


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

### What this hands to the model week

1. **The vector is frozen and audited** — any model built on it inherits four passed attacks rather
   than a fresh argument about what was allowed.
2. **`clicks_31d` stays out.** The ML-07 rule may use clicks; a model may not. That is the harder
   bar, and the honest one.
3. **Per-client normalisation is the next experiment**, backed twice: ML-06's 10x client spread and
   ML-04's negative R². `impressions_share_of_client` is the first step; normalising the *target*
   per client is the next.